# Demo 7: Tilastollinen testaus

## Ohjeet

Tässä demossa harjoitellaan tilastollista testausta Pythonilla. Käytettävä aineisto on Excel-tiedostossa Tilastollinen_testaus.xlsx.

## Mitä tilastollinen testaus tarkoittaa?

Tilastollisella testauksella tutkitaan, voidaanko aineistossa havaittua eroa tai yhteyttä pitää tilastollisesti merkitsevänä vai voiko havainto selittyä satunnaisvaihtelulla.

Testauksessa muodostetaan kaksi hypoteesia:

- **Nollahypoteesi (H₀):** eroa tai yhteyttä ei ole.
- **Vastahypoteesi (H₁):** ero tai yhteys on olemassa.

Testin tuloksena saatava **p-arvo** auttaa arvioimaan, onko aineistossa riittävästi näyttöä nollahypoteesia vastaan.

Tässä demossa käytetään merkitsevyystasoa **0,05**.

- Jos **p < 0,05**, tulos on tilastollisesti merkitsevä ja nollahypoteesi hylätään.
- Jos **p ≥ 0,05**, tulos ei ole tilastollisesti merkitsevä eikä nollahypoteesia voida hylätä.

Jokaisesta testistä tulee osata kertoa:

- mitä tutkittiin
- mitä ryhmiä tai muuttujia verrattiin
- mikä oli testin p-arvo
- oliko tulos tilastollisesti merkitsevä
- mitä tuloksesta voidaan päätellä

Pelkkä p-arvon ilmoittaminen ei riitä. Tuloksesta pitää kertoa omin sanoin, mitä aineiston perusteella voidaan päätellä.

Muista myös, että tilastollisesti merkitsevä tulos ei välttämättä tarkoita, että havaittu ero tai yhteys olisi käytännössä olemassa. Tätä pitää erikseen pohtia. 



## 1. Aineistoon tutustuminen

Ennen tilastollisten testien tekemistä aineistoon pitää tutustua. On tärkeää tietää, millaisia muuttujia aineistossa on ja millaisia arvoja ne sisältävät.

**Tehtävät**

- Lue aineisto Excel-tiedostosta.
- Tarkastele aineiston rakennetta, muuttujia ja ensimmäisiä havaintoja.
- Tutki numeeristen muuttujien keskeisiä tunnuslukuja.
- Tarkista kategoristen muuttujien arvot.
- Huomaa, että tyytyväisyys on Likert-asteikollinen muuttuja: 1 = erittäin tyytymätön ja 5 = erittäin tyytyväinen.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Luetaan aineisto Excel-tiedostosta.
df = pd.read_excel("shared/Tilastollinen_testaus.xlsx", sheet_name="Aineisto")

# Korjataan aineiston ensimmäisen sarakkeen nimi selkeämmäksi.
df = df.rename(columns={df.columns[0]: "Opiskelija"})

# Näytetään aineiston viisi ensimmäistä riviä.
df.head()

In [ ]:
# Tarkastellaan aineiston kokoa, sarakkeita ja tietotyyppejä.
print("Aineiston koko:", df.shape)
print("\nSarakkeet ja tietotyypit:")
df.info()

In [ ]:
# Numeeristen muuttujien keskeiset tunnusluvut.
df[["Koepisteet", "Tyytyväisyys", "Opiskelutunnit"]].describe()

In [ ]:
# Tarkistetaan kategoristen muuttujien arvot.
for sarake in ["Ryhmä", "Opiskelumuoto", "Läpäissyt"]:
    print(f"\n{sarake}:")
    print(df[sarake].value_counts())

## 2. Kahden ryhmän keskiarvojen vertailu: t-testi

**T-testillä** tutkitaan, eroavatko kahden riippumattoman ryhmän keskiarvot toisistaan.

Tässä vertaamme kevään ja syksyn opiskelijoiden koepisteitä. Havainnot ovat riippumattomia, koska sama opiskelija ei ole molemmissa ryhmissä.

Ennen testiä tarkastelemme ryhmien havaintomääriä, keskiarvoja, keskihajontoja ja jakaumia. Käytämme **Welchin t-testiä**, joka ei oleta ryhmien varianssien olevan yhtä suuret.

- **H₀:** Kevään ja syksyn koepisteiden keskiarvot eivät eroa toisistaan.
- **H₁:** Kevään ja syksyn koepisteiden keskiarvot eroavat toisistaan.

**Tehtävät**

- Laske kevään ja syksyn koepisteiden keskiarvot ja keskihajonnat.
- Tarkastele koepisteiden jakaumia histogrammilla ja laatikkokaaviolla.
- Arvioi t-testin käyttöedellytyksiä.
- Tee riippumattomien otosten t-testi.
- Tarkastele p-arvoa.
- Tee testin perusteella johtopäätös.

In [ ]:
# Poimitaan kevään ja syksyn koepisteet omiin sarjoihinsa.
kevat_pisteet = df.loc[df["Ryhmä"] == "Kevät 2026", "Koepisteet"]
syksy_pisteet = df.loc[df["Ryhmä"] == "Syksy 2026", "Koepisteet"]

# Tulostetaan ryhmien havaintomäärät, keskiarvot ja keskihajonnat.
yhteenveto = df.groupby("Ryhmä")["Koepisteet"].agg(
    Havaintoja="count",
    Keskiarvo="mean",
    Keskihajonta="std"
)

yhteenveto.round(2)

In [ ]:
# Piirretään koepisteiden jakaumat histogrammeina.
sns.histplot(data=df, x="Koepisteet", hue="Ryhmä", kde=True, element="step")
plt.title("Koepisteiden jakaumat keväällä ja syksyllä")
plt.xlabel("Koepisteet")
plt.ylabel("Lukumäärä")
plt.show()

# Piirretään ryhmien koepisteet laatikkokaaviona.
sns.boxplot(data=df, x="Ryhmä", y="Koepisteet")
plt.title("Koepisteet ryhmittäin")
plt.xlabel("Ryhmä")
plt.ylabel("Koepisteet")
plt.show()

In [ ]:
# Tehdään Welchin t-testi.
t_arvo, p_arvo = stats.ttest_ind(kevat_pisteet, syksy_pisteet, equal_var=False)

print(f"t-arvo: {t_arvo:.3f}")
print(f"p-arvo: {p_arvo:.4f}")

if p_arvo < 0.05:
    print("Johtopäätös: Kevään ja syksyn koepisteiden keskiarvoissa on tilastollisesti merkitsevä ero.")
else:
    print("Johtopäätös: Kevään ja syksyn koepisteiden keskiarvoissa ei havaittu tilastollisesti merkitsevää eroa.")

## 3. Tyytyväisyyden vertailu: Mann–Whitneyn U-testi

**Mann–Whitneyn U-testi** on kahden riippumattoman ryhmän vertailuun tarkoitettu testi. Se ei edellytä normaalijakaumaa samalla tavalla kuin t-testi.

Testi sopii hyvin Likert-asteikollisen tyytyväisyyden vertailuun. Ryhmiä kuvataan tässä mediaanilla.

Teemme kolme vertailua:

1. Kevään ja syksyn verkko-opiskelijat.
2. Kevään ja syksyn lähiopiskelijat.
3. Koko vuoden verkko- ja lähiopiskelijat.

Jokaisessa vertailussa:

- **H₀:** Ryhmien tyytyväisyydessä ei ole eroa.
- **H₁:** Ryhmien tyytyväisyydessä on ero.

**Tehtävät**

- Laske verrattavien ryhmien mediaanit.
- Tee Mann–Whitneyn U-testit.
- Tarkastele testien p-arvoja.
- Tee jokaisesta vertailusta johtopäätös.

In [ ]:
# Apufunktio tekee Mann–Whitneyn U-testin ja tulostaa tuloksen.
def mann_whitney_vertailu(ryhma1, ryhma2, nimi1, nimi2):
    u_arvo, p_arvo = stats.mannwhitneyu(ryhma1, ryhma2, alternative="two-sided")

    print(f"{nimi1}: mediaani = {ryhma1.median():.1f}, n = {len(ryhma1)}")
    print(f"{nimi2}: mediaani = {ryhma2.median():.1f}, n = {len(ryhma2)}")
    print(f"U-arvo: {u_arvo:.3f}")
    print(f"p-arvo: {p_arvo:.4f}")

    if p_arvo < 0.05:
        print("Johtopäätös: Ryhmien tyytyväisyydessä on tilastollisesti merkitsevä ero.")
    else:
        print("Johtopäätös: Ryhmien tyytyväisyydessä ei havaittu tilastollisesti merkitsevää eroa.")

In [ ]:
# 1. Kevään ja syksyn verkko-opiskelijoiden tyytyväisyys.
kevat_verkko = df.loc[
    (df["Ryhmä"] == "Kevät 2026") & (df["Opiskelumuoto"] == "Verkko"),
    "Tyytyväisyys"
]
syksy_verkko = df.loc[
    (df["Ryhmä"] == "Syksy 2026") & (df["Opiskelumuoto"] == "Verkko"),
    "Tyytyväisyys"
]

mann_whitney_vertailu(
    kevat_verkko,
    syksy_verkko,
    "Kevään verkko-opiskelijat",
    "Syksyn verkko-opiskelijat"
)

In [ ]:
# 2. Kevään ja syksyn lähiopiskelijoiden tyytyväisyys.
kevat_lahi = df.loc[
    (df["Ryhmä"] == "Kevät 2026") & (df["Opiskelumuoto"] == "Lähi"),
    "Tyytyväisyys"
]
syksy_lahi = df.loc[
    (df["Ryhmä"] == "Syksy 2026") & (df["Opiskelumuoto"] == "Lähi"),
    "Tyytyväisyys"
]

mann_whitney_vertailu(
    kevat_lahi,
    syksy_lahi,
    "Kevään lähiopiskelijat",
    "Syksyn lähiopiskelijat"
)

In [ ]:
# 3. Koko vuoden verkko- ja lähiopiskelijoiden tyytyväisyys.
verkko = df.loc[df["Opiskelumuoto"] == "Verkko", "Tyytyväisyys"]
lahi = df.loc[df["Opiskelumuoto"] == "Lähi", "Tyytyväisyys"]

mann_whitney_vertailu(
    verkko,
    lahi,
    "Verkko-opiskelijat",
    "Lähiopiskelijat"
)

## 4. Opiskelumuodon ja läpäisyn yhteys: χ²-testi

**χ²-riippumattomuustestillä** tutkitaan kahden kategorisen muuttujan välistä yhteyttä.

Tässä tutkimme, onko opiskelumuodolla yhteyttä kurssin läpäisyyn. Emme vertaile keskiarvoja, vaan tarkastelemme eri ryhmiin kuuluvien opiskelijoiden lukumääriä.

- **H₀:** Opiskelumuodon ja läpäisyn välillä ei ole yhteyttä.
- **H₁:** Opiskelumuodon ja läpäisyn välillä on yhteys.

**Tehtävät**

- Muodosta opiskelumuodosta ja läpäisystä ristiintaulukko.
- Laske läpäisyosuudet verkko- ja lähiopiskelijoille.
- Tee χ²-riippumattomuustesti.
- Tarkastele testin p-arvoa ja odotettuja frekvenssejä.
- Tee testin perusteella johtopäätös.

In [ ]:
# Muodostetaan ristiintaulukko havaituista lukumääristä.
ristiintaulukko = pd.crosstab(df["Opiskelumuoto"], df["Läpäissyt"])
ristiintaulukko

In [ ]:
# Lasketaan riviprosentit, jotta ryhmien läpäisyosuuksia voidaan verrata.
riviprosentit = pd.crosstab(
    df["Opiskelumuoto"],
    df["Läpäissyt"],
    normalize="index"
) * 100

riviprosentit.round(1)

In [ ]:
# Tehdään χ²-riippumattomuustesti.
chi2_arvo, p_arvo, vapausasteet, odotetut = stats.chi2_contingency(ristiintaulukko)

print(f"χ²-arvo: {chi2_arvo:.3f}")
print(f"Vapausasteet: {vapausasteet}")
print(f"p-arvo: {p_arvo:.4f}")

# Muutetaan odotetut frekvenssit taulukoksi.
odotetut_df = pd.DataFrame(
    odotetut,
    index=ristiintaulukko.index,
    columns=ristiintaulukko.columns
)

print("\nOdotetut frekvenssit:")
display(odotetut_df.round(2))

if p_arvo < 0.05:
    print("Johtopäätös: Opiskelumuodon ja läpäisyn välillä on tilastollisesti merkitsevä yhteys.")
else:
    print("Johtopäätös: Opiskelumuodon ja läpäisyn välillä ei havaittu tilastollisesti merkitsevää yhteyttä.")

## 5. Opiskelutuntien ja koepisteiden yhteys: Pearsonin korrelaatiotesti

**Korrelaatiolla** tutkitaan kahden numeerisen muuttujan välistä yhteyttä.

Pearsonin korrelaatiokerroin saa arvon väliltä −1 ja 1:

- Positiivinen arvo tarkoittaa, että toisen muuttujan kasvaessa myös toinen yleensä kasvaa.
- Negatiivinen arvo tarkoittaa, että toisen muuttujan kasvaessa toinen yleensä pienenee.
- Lähellä nollaa oleva arvo kertoo heikosta lineaarisesta yhteydestä.

Pearsonin korrelaatio tarkastelee lineaarista yhteyttä. Siksi muuttujien yhteyttä tarkastellaan ensin hajontakuvion avulla.

- **H₀:** Opiskelutuntien ja koepisteiden välillä ei ole lineaarista yhteyttä.
- **H₁:** Opiskelutuntien ja koepisteiden välillä on lineaarinen yhteys.

**Tehtävät**

- Piirrä opiskelutunneista ja koepisteistä hajontakuvio.
- Arvioi kuvaajan perusteella yhteyden suuntaa ja voimakkuutta.
- Laske Pearsonin korrelaatiokerroin.
- Testaa korrelaation tilastollinen merkitsevyys.
- Tarkastele korrelaatiokerrointa ja p-arvoa.
- Tee testin perusteella johtopäätös.
- Pohdi, voidaanko korrelaation perusteella päätellä syy-seuraussuhdetta.

In [ ]:
# Piirretään hajontakuvio ja regressiosuora.
sns.regplot(
    data=df,
    x="Opiskelutunnit",
    y="Koepisteet",
    scatter_kws={"alpha": 0.7},
    line_kws={"color": "red"}
)
plt.title("Opiskelutuntien ja koepisteiden yhteys")
plt.xlabel("Opiskelutunnit viikossa")
plt.ylabel("Koepisteet")
plt.show()

In [ ]:
# Lasketaan Pearsonin korrelaatiokerroin ja sen p-arvo.
r_arvo, p_arvo = stats.pearsonr(df["Opiskelutunnit"], df["Koepisteet"])

print(f"Pearsonin korrelaatiokerroin: {r_arvo:.3f}")
print(f"p-arvo: {p_arvo:.4f}")

if p_arvo < 0.05:
    print("Johtopäätös: Opiskelutuntien ja koepisteiden välillä on tilastollisesti merkitsevä lineaarinen yhteys.")
else:
    print("Johtopäätös: Opiskelutuntien ja koepisteiden välillä ei havaittu tilastollisesti merkitsevää lineaarista yhteyttä.")

print("Korrelaatio ei yksin osoita syy-seuraussuhdetta.")